In [ ]:
# Cell 1: Import libraries and setup

import os
import sys
import yaml
from nuscenes.nuscenes import NuScenes
import matplotlib.pyplot as plt

# Add project root to sys.path
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added project root to sys.path: {PROJECT_ROOT}")

# Import utility modules
from utils.refactor import (
    find_instances_in_scene,
    get_lidar_sweeps_for_interval
)

from utils.refactor import create_multi_box_exact_synchronized_animation

# Load config
try:
    with open(os.path.join(PROJECT_ROOT, 'config/m_detector_config.yaml'), 'r') as f:
        config = yaml.safe_load(f)
except FileNotFoundError:
    print("Config file not found. Using default settings.")
    config = {'nuscenes': {'version': 'v1.0-mini', 'dataroot': '/path/to/nuscenes'}}

# Initialize NuScenes
nusc = NuScenes(
    version=config['nuscenes']['version'],
    dataroot=config['nuscenes']['dataroot'],
    verbose=True
)

print("NuScenes and helper functions loaded.")

In [ ]:
# Cell 2: Select a scene and get lidar sweeps with annotations

scene_index = 1  # Change as needed
my_scene = nusc.scene[scene_index]
print(f"Selected Scene: {my_scene['name']} (Description: {my_scene['description']})")

all_lidar_sweeps = get_lidar_sweeps_for_interval(nusc, my_scene['first_sample_token'], my_scene['last_sample_token'])
print(f"Found {len(all_lidar_sweeps)} LiDAR sweeps in the scene")

instances = find_instances_in_scene(nusc, my_scene['token'], min_annotations=2)
print(f"Found {len(instances)} instances")

In [ ]:
# Cell 3: Create animation
plt.rcParams['animation.embed_limit'] = 2**128 # allow large animation

# Create animation with multiple boxes
multi_box_animation_html, anim = create_multi_box_exact_synchronized_animation(
    nusc, instances, all_lidar_sweeps,
    interval_ms=50, # Corresponds to 20 FPS for display
    save_path="output_animation.mp4",
    save_writer="ffmpeg",
    save_fps=20, # Save at 20 FPS
    save_dpi=200,
    point_downsample=1
)



In [ ]:
# Cell 4: Display

if multi_box_animation_html:
    display(multi_box_animation_html)